# EDA — Engagement marketplace
Analyse exploratoire du train/test. Le pipeline complet est dans `build_features.py` → `compare_models.py` → `train_final.py`.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
tr = pd.read_parquet('data/train.parquet'); te = pd.read_parquet('data/test.parquet')
print(tr.shape, te.shape); tr.head()

## Cible
Taux de positifs ≈ 16.6% : classes déséquilibrées mais l'AUC n'y est pas sensible.

In [ ]:
tr.label.value_counts(normalize=True)

## Valeurs manquantes
Les ids (`user_id` 34%, `item_id`/`session_id` ~10%) ont des manquants injectés au hasard, au même taux dans train et test. Les manquants de `size`/`category_l3` sont structurels (catégories sans taille).

In [ ]:
pd.DataFrame({'train': tr.isna().mean(), 'test': te.isna().mean()}).round(4)

## Recouvrement train/test
Même période et quasi mêmes users/sessions : le test est un échantillon aléatoire de lignes. On peut donc exploiter les labels des autres lignes d'une même session/d'un même user.

In [ ]:
print('période train', tr.event_time.min(), tr.event_time.max())
print('période test ', te.event_time.min(), te.event_time.max())
for c in ['user_id','session_id','item_id']:
    print(c, 'part des lignes test dont l\'id est vu dans le train:', round(te[c].dropna().isin(set(tr[c].dropna())).mean(), 3))

## Taux d'engagement par variable

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
for a, c in zip(ax.flat, ['category_l1','condition','shipping_method','sequence_length']):
    tr.groupby(c).label.mean().plot.bar(ax=a, title=c)
tr.groupby(pd.qcut(tr.price, 10), observed=True).label.mean().plot.bar(ax=ax.flat[4], title='prix (déciles)')
tr.groupby(tr.event_time.dt.hour).label.mean().plot(ax=ax.flat[5], title='heure')
plt.tight_layout()

## Effet session
Les labels sont très groupés par session : la plupart des sessions sont 0 partout, d'autres ont des séries de 1. C'est le signal principal exploité par les features leave-one-out.

In [ ]:
s = tr.groupby('session_id').label.agg(['mean','size'])
s[s['size']>=5]['mean'].hist(bins=50); plt.title('taux d\'engagement par session (sessions ≥5 lignes)')

## Prix
Distribution très asymétrique → log(prix) ; plus l'item est cher, moins il est engagé.

In [ ]:
np.log1p(tr.price).hist(bins=100); plt.title('log(1+prix)')